In [ ]:
#@title Physics-Guided Battery XAI — one-cell Colab runner
# Paste this single cell into Google Colab Pro+ and run it. If the notebook was
# opened directly from GitHub, set REPO_URL to your repository URL first.
REPO_URL = ""  #@param {type:"string"}
BRANCH = ""    #@param {type:"string"}
PROJECT_DIR = "/content/battery-xai-physics"  #@param {type:"string"}
EXPERIMENT_NAME = "colab_single_cell_run"  #@param {type:"string"}
CONFIG_PATH = "configs/colab_single_cell.yaml"  #@param {type:"string"}
USE_DEMO_DATA_IF_MISSING = True  #@param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = True  #@param {type:"boolean"}
INSTALL_FULL_STACK = True  #@param {type:"boolean"}

import os, sys, subprocess, textwrap
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

project = Path(PROJECT_DIR)
if not (project / "setup.py").exists():
    if not REPO_URL:
        raise RuntimeError(textwrap.dedent(f"""
        Repository files are not present at {PROJECT_DIR}.
        Set REPO_URL at the top of this single cell, for example:
            REPO_URL = "https://github.com/<your-user>/battery-xai-physics.git"
        Then rerun this cell. If you uploaded the repository manually, set PROJECT_DIR
        to the folder containing setup.py.
        """))
    clone_cmd = ["git", "clone"]
    if BRANCH:
        clone_cmd += ["--branch", BRANCH]
    clone_cmd += [REPO_URL, str(project)]
    subprocess.check_call(clone_cmd)

os.chdir(project)
print(f"Running from: {Path.cwd()}")

install_target = ".[full]" if INSTALL_FULL_STACK else "."
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", install_target])

import yaml
cfg_path = Path(CONFIG_PATH)
with cfg_path.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)
cfg.setdefault("data", {})["demo_if_missing"] = bool(USE_DEMO_DATA_IF_MISSING)
# Drive is already mounted above; avoid a second mount prompt from the runner.
cfg.setdefault("project", {}).setdefault("colab", {})["mount_drive"] = False
runtime_cfg = Path("configs/_colab_runtime.yaml")
with runtime_cfg.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

from battery_xai.experiments.run_pipeline import run
outputs = run(str(runtime_cfg), EXPERIMENT_NAME)

print("\n✅ Battery XAI pipeline completed. Artifacts:")
for key, value in outputs.items():
    print(f"  {key}: {value}")

try:
    import pandas as pd
    display(pd.read_csv(Path(outputs["results"]) / "transfer_results.csv"))
except Exception as exc:
    print(f"Transfer table preview skipped: {exc}")

try:
    from IPython.display import Image, display
    figure = Path(outputs.get("figures", "figures")) / "capacity_fade.png"
    if figure.exists():
        display(Image(filename=str(figure)))
except Exception as exc:
    print(f"Figure preview skipped: {exc}")
